# Payload validation

Checks that the request template describes the train we think it does, before
any collection spends an hour on it. Run this whenever anything in
`trassenfinder.PAYLOAD_TEMPLATE` changes, and any time a collected number looks
wrong.

## Why this notebook exists

The 2026-08-30 speed sweep produced a result that did not add up. Allowing
high-speed lines rerouted the train onto genuinely shorter paths — one segment
went from 203.2 to 175.7 km — and realised average speed moved from 97.0 to
100.2 km/h. A train booked at 200 km/h, on track rated 250-300, averaging 100.

The ceiling was not the line. And it sat at exactly 120 km/h booked: every grid
point above 120 returned byte-identical results in both strata.

Two fields in the template are the obvious suspects:

```python
"bremshundertstel": 70,
"bremsstellung": "P",
```

**Bremsstellung P is valid to 120 km/h**; R is required above that.
**70 Bremshundertstel** is a freight-grade braked-weight percentage — a
locomotive-hauled passenger consist on disc brakes is far higher. Either alone
would cap the train near 120.

The rest of the template's defaults point the same way: `wagenzugmasse_t: 1200`,
`wagenzuglaenge_m: 600`, `hauptnummer: "6185"` (a freight locomotive), and
`v_max: 100`. Those four are all overwritten per request so they do no harm, but
they say the template was adapted from a freight example by changing
`verkehrsart` to `spfv_lok` and little else. The braking fields were not part of
that adaptation.

`01`'s own markdown recorded the assumption that let this through: these fields
are *"fixed for all compositions, so they act as a level effect rather than a
difference between trains."* True, and not the same as harmless. A constant that
caps achievable speed is exactly what breaks a study of speed.

## What this notebook does not assume

Sections 2 and 3 test the two fields directly rather than arguing from the
regulations. Section 1 dumps a full response so the API can be read rather than
inferred from. Section 4 puts the running times in front of a human who knows
what German rail timings should look like.

## 1. What does the API actually return?

`query()` keeps three fields out of the response and discards everything else.
Whatever Trassenfinder says about *how* it computed the run — per-section
speeds, applied limits, the vehicle it resolved — is in there and has never been
looked at.

This writes one full response to `data/raw_response_sample.json` and walks the
structure. Read the summary block and the first route point closely: if the API
reports a limiting speed anywhere, that answers the question outright.

In [ ]:
import json

import pandas as pd

import trassenfinder as tf
from data_sources import DATA_DIR, source_input

DATA_DIR.mkdir(parents=True, exist_ok=True)

compositions = pd.read_csv(source_input("compositions.csv"))
composition = compositions[compositions["composition_id"] == "REF-BAL-9"].iloc[0]

# Hamburg Hbf -> Hannover Hbf. A main-line, electrified, high-capacity corridor
# with no gradient worth mentioning: if a locomotive-hauled train is slow here,
# it is the train.
START, END = "AH", "HH"

tf.resolve_stations([START, END], composition)
raw = tf.query_raw(START, END, composition, v_max_kmh=200)

with open(DATA_DIR / "raw_response_sample.json", "w", encoding="utf-8") as fh:
    json.dump(raw, fh, indent=2, ensure_ascii=False)
print(f"full response -> {DATA_DIR / 'raw_response_sample.json'}\n")


def walk(node, prefix="", depth=0, max_depth=2):
    """Print the shape of the response without dumping every route point."""
    if depth > max_depth:
        return
    if isinstance(node, dict):
        for key, value in node.items():
            kind = type(value).__name__
            extra = (
                f" [{len(value)}]"
                if isinstance(value, (list, dict))
                else f" = {value!r}"
            )
            print(f"{'  ' * depth}{prefix}{key}: {kind}{extra[:70]}")
            if isinstance(value, (dict, list)):
                walk(value, "", depth + 1, max_depth)
    elif isinstance(node, list) and node:
        walk(node[0], "[0].", depth, max_depth)


walk(raw)

route = raw["result"]["gewichtete_route"]
print("\n--- zusammenfassung, in full ---")
for key, value in route["zusammenfassung"].items():
    print(f"  {key}: {value}")

print("\n--- first route point, in full ---")
print(json.dumps(route["routenpunkte"][0], indent=2, ensure_ascii=False)[:2500])

# Anything mentioning speed, anywhere in the response, is worth seeing.
blob = json.dumps(raw, ensure_ascii=False)
print("\n--- keys containing a speed-ish word ---")
seen = set()


def find_keys(node):
    if isinstance(node, dict):
        for key, value in node.items():
            if any(
                w in key.lower() for w in ("geschw", "v_max", "vmax", "speed", "brems")
            ):
                if key not in seen:
                    seen.add(key)
                    print(f"  {key}: {value!r}"[:160])
            find_keys(value)
    elif isinstance(node, list):
        for item in node[:3]:
            find_keys(item)


find_keys(raw)

## 2. Bremsstellung

The direct test. One segment, one train, `v_max` booked at 200 throughout, and
only the brake position changed.

If **P** and **R** return the same running time, the brake position is not what
limits the train and the hypothesis is wrong — go back to section 1's dump.

If **R** is materially faster, the collected dataset describes a train braked
for 120 km/h operation, and `samples_all` needs recollecting rather than
patching.

In [ ]:
def run(label, **overrides):
    """One query, reduced to the numbers that matter here."""
    try:
        result = tf.query(START, END, composition, v_max_kmh=200, **overrides)
    except (tf.TrassenfinderError, ValueError) as exc:
        return {"variant": label, "error": str(exc)[:90]}

    return {
        "variant": label,
        "distance_km": result["distance_km"],
        "travel_time_min": result["travel_time_min"],
        "avg_speed_kmh": round(
            result["distance_km"] / (result["travel_time_min"] / 60), 1
        ),
        "energy_kwh": result["energy_kwh"],
    }


rows = [
    run("P (template)", zug={"bremsstellung": "P"}),
    run("R", zug={"bremsstellung": "R"}),
]

brake = pd.DataFrame(rows)
print(brake.to_string(index=False))

if "avg_speed_kmh" in brake.columns and brake["avg_speed_kmh"].notna().all():
    p_speed, r_speed = brake["avg_speed_kmh"]
    print(f"\nR is {r_speed - p_speed:+.1f} km/h faster than P")
    if r_speed - p_speed > 5:
        print(
            "  CONFIRMED: the brake position limits the train. Everything "
            "collected so far describes a train braked for 120 km/h."
        )
    else:
        print(
            "  Not the brake position on its own. Try section 3 before "
            "concluding anything."
        )

## 3. Bremshundertstel

The braked-weight percentage sets the permitted speed through the brake tables,
so it caps the train independently of the brake position. A locomotive-hauled
consist on disc brakes sits far above 70.

Swept with the brake position held at R, so this isolates the second field.
Watch where running time stops improving: that is the point at which something
else becomes the binding constraint, and it tells you what a realistic value
buys you.

In [ ]:
BRH_GRID = [70, 90, 110, 130, 150, 180]

brh = pd.DataFrame(
    [
        run(f"R, {value} BrH", zug={"bremsstellung": "R", "bremshundertstel": value})
        for value in BRH_GRID
    ]
)
print(brh.to_string(index=False))

if "avg_speed_kmh" in brh.columns and brh["avg_speed_kmh"].notna().all():
    print("\nSpeed gained by each step:")
    print(brh.set_index("variant")["avg_speed_kmh"].diff().round(2).to_string())
    print(
        f"\nBest: {brh['avg_speed_kmh'].max():.1f} km/h against "
        f"{brh['avg_speed_kmh'].iloc[0]:.1f} at the template's 70 BrH"
    )

# Both fields together, against the template as collected.
combined = pd.DataFrame(
    [
        run("template (P, 70)", zug={"bremsstellung": "P", "bremshundertstel": 70}),
        run("corrected (R, 150)", zug={"bremsstellung": "R", "bremshundertstel": 150}),
    ]
)
print("\n--- template vs corrected ---")
print(combined.to_string(index=False))

if "energy_kwh" in combined.columns and combined["energy_kwh"].notna().all():
    old_e, new_e = combined["energy_kwh"]
    old_t, new_t = combined["travel_time_min"]
    print(f"\nenergy  {old_e} -> {new_e} kWh  ({(new_e / old_e - 1) * 100:+.1f}%)")
    print(f"time    {old_t} -> {new_t} min  ({(new_t / old_t - 1) * 100:+.1f}%)")
    print(
        "\nA large energy change here means the calibration coefficients move "
        "too, not just the speed range they are valid over."
    )

## 4. Do the running times look like German railway?

The check that needs no regulations and no API internals: put the numbers in
front of someone who knows the corridors.

`travel_time_min` is **technical running time** — no dwell, no recovery margin,
no timetable padding — so it should come out *faster* than a published schedule,
not slower. If a main-line corridor is coming back slower than the real
timetable, the train being modelled is not the train intended, whatever the
cause turns out to be.

Hamburg–Hannover is the sharpest test in the set: 178 km of flat, four-track,
200 km/h main line. Anything locomotive-hauled and properly braked should be
well under 90 minutes of pure running time.

In [ ]:
CHECKS = [
    ("AH", "HH", "Hamburg Hbf - Hannover Hbf"),
    ("HH", "BLS", "Hannover Hbf - Berlin Hbf"),
    ("FF", "MH", "Frankfurt Hbf - Munich Hbf"),
    ("KK", "FF", "Koeln Hbf - Frankfurt Hbf"),
]

for start, end, _ in CHECKS:
    tf.resolve_stations([start, end], composition)

reality = []
for start, end, name in CHECKS:
    for label, overrides in (
        ("template (P, 70)", {"bremsstellung": "P", "bremshundertstel": 70}),
        ("corrected (R, 150)", {"bremsstellung": "R", "bremshundertstel": 150}),
    ):
        try:
            result = tf.query(start, end, composition, v_max_kmh=200, zug=overrides)
            reality.append(
                {
                    "corridor": name,
                    "variant": label,
                    "distance_km": result["distance_km"],
                    "running_time_min": result["travel_time_min"],
                    "avg_speed_kmh": round(
                        result["distance_km"] / (result["travel_time_min"] / 60), 1
                    ),
                }
            )
        except (tf.TrassenfinderError, ValueError) as exc:
            reality.append(
                {
                    "corridor": name,
                    "variant": label,
                    "distance_km": None,
                    "running_time_min": None,
                    "avg_speed_kmh": None,
                    "error": str(exc)[:70],
                }
            )

reality_df = pd.DataFrame(reality)
print(reality_df.to_string(index=False))

print(
    "\nCompare each running time against what you know the corridor takes. "
    "Technical running time carries no dwell and no recovery margin, so it "
    "should beat the published schedule. If it does not, the payload is wrong."
)

## 5. Are the derived values right?

`trassenfinder.py` now derives `bremshundertstel` and `streckenklasse` per
composition instead of carrying constants. This checks the derivation against
the API rather than against the arithmetic that produced it.

Three things to see:

1. **The derived values themselves**, with the axle and metre loads they came
   from, so the line class can be audited against EN 15528 without rerunning
   anything.
2. **Whether more braking would still buy speed.** If pushing BrH above the
   derived value keeps improving the running time, the fleet assumption is
   conservative and worth revisiting. If it flattens, the derivation is at or
   past the point where something else binds.
3. **Whether D2 opens routes D4 closed.** Expected to be a small effect: the
   locomotive's 21.75 t axle load excludes C-class lines whatever the digit.

In [ ]:
print("Derived per composition")
print(f"{'composition':14s} {'BrH':>4s}  {'class':5s} {'t/axle':>7s} {'t/m':>6s}")
for _, comp in compositions.iterrows():
    wagenzug_t = comp["coaches_gross_weight_80pct_t_wagenzugmasse"]
    wagenzug_m = comp["coaches_length_m_wagenzuglaenge"]
    axle = max(tf.LOCO_MASS_T / tf.LOCO_AXLES, wagenzug_t / comp["n_coaches"] / 4)
    metre = max(tf.LOCO_MASS_T / tf.LOCO_LENGTH_M, wagenzug_t / wagenzug_m)
    print(
        f"{comp['composition_id']:14s} {tf.bremshundertstel(comp):4d}  "
        f"{tf.streckenklasse(comp):5s} {axle:7.2f} {metre:6.2f}"
    )

derived_brh = tf.bremshundertstel(composition)

print(f"\n--- does more braking than {derived_brh} BrH still buy speed? ---")
print(
    pd.DataFrame(
        [
            run(f"{value} BrH", zug={"bremshundertstel": value})
            for value in (
                derived_brh - 40,
                derived_brh - 20,
                derived_brh,
                derived_brh + 20,
                derived_brh + 40,
            )
        ]
    ).to_string(index=False)
)

print("\n--- does D2 open routes that D4 closed? ---")
# A regional pair, the category that failed most often under D4.
PROBE = [("AH", "HH"), ("TSG", "TDIH"), ("MURB", "RBSS")]
for start, end in PROBE:
    tf.resolve_stations([start, end], composition)
    for klasse in ("D4", "D2"):
        try:
            result = tf.query(start, end, composition, zug={"streckenklasse": klasse})
            print(
                f"  {start:6s} -> {end:6s}  {klasse}: "
                f"{result['distance_km']:7.1f} km  "
                f"{result['travel_time_min']:4d} min"
            )
        except (tf.TrassenfinderError, ValueError) as exc:
            print(f"  {start:6s} -> {end:6s}  {klasse}: {str(exc)[:60]}")

## 6. Verdict

Write the answer down here before touching anything else, because what follows
depends on it.

The 2026-08-30 run confirmed the braking fields, so every sample collected
before that date — `samples_all` included, not only the sweep — describes a
train limited to about 110 km/h. That was not a sweep problem to patch:

1. `bremshundertstel` and `streckenklasse` are now derived per composition and
   `bremsstellung` is R. See `calib/README.md` for the basis of each.
2. Re-run `01` in full. Roughly 1,560 requests, 30-50 minutes. The old
   `samples_*.csv` are superseded, not amendable.
3. Re-run `01b`. The grid moved with the ceiling.
4. Re-run `02`. Every coefficient moves.

If a future run shows something still capping the train, section 1's dump is the
place to look — specifically the resolved vehicle. `hauptnummer` 6193 with
`unternummer` 2 and `kennung_wert` 80 has never been verified against
Trassenfinder's own vehicle database, and a variant carrying a lower stored
maximum speed would cap the train the same way.

The general lesson belongs in the decisions log regardless: a request field
being constant across every row of a collection makes it invisible in the data,
not harmless. Constants set the level, and a constant that caps speed silently
bounds what the whole dataset can ever say. That is what this notebook exists to
catch, and why it runs before a collection rather than after one.